
# ⚽ Operaciones con DataFrames en Apache Spark

Este notebook contiene ejemplos prácticos de las operaciones más comunes que se pueden realizar con DataFrames en Spark, utilizando datos simulados del ámbito deportivo.


Comenzamos importando las librerías necesarias y creando una sesión de Spark.

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Operaciones con DataFrames").getOrCreate()

In [0]:

data = [
    ("Pedri", "FC Barcelona", 32.5, 1, 85, 78, 90),
    ("Modric", "Real Madrid", 30.1, 2, 95, 88, 87),
    ("Kroos", "Real Madrid", 29.8, 1, 102, 92, 90),
    ("Gavi", "FC Barcelona", 31.2, 0, 76, 65, 80),
    ("Kubo", "Real Sociedad", 33.0, 3, 88, 70, 85),
    ("Isco", "Real Betis", 28.7, 2, 90, 75, 83)
]

columns = ["Jugador", "Equipo", "Velocidad", "Goles", "Pases", "PasesCompletados", "Minutos"]

df = spark.createDataFrame(data, columns)
df.show()


## 🔍 Selección y filtrado

In [0]:
# Seleccionar columnas específicas
df.select("Jugador", "Goles", "Equipo").show()

In [0]:
# Filtrar jugadores con más de 1 gol
df.filter(df.Goles > 1).show()

In [0]:
# Eliminar columna de velocidad.
df.drop("Velocidad").show()

## 🔄 Transformación de columnas

In [0]:
from pyspark.sql.functions import avg, sum, count, min, max, col

# Crear columna de precisión de pase (%)
df.withColumn("PrecisionPase", (col("PasesCompletados") / col("Pases") * 100)).show()

In [0]:
# Renombrar columna de minutos
df.withColumnRenamed("Minutos", "MinutosJugados").show()

In [0]:
# Cambiar tipo de dato de goles a string (nueva columna)
df.withColumn("Goles_str", col("Goles").cast("string")).printSchema()

## 📊 Agregación y agrupamiento

La función `agg()` agrega los resultados agrupados por la función `groupBy()`. Esta función recibe como parámetros una o varias funciones de agregación que resultarán en nuevas columnas.

In [0]:
from pyspark.sql.functions import avg, sum, count, min, max

df.groupBy("Equipo").agg(
    avg("Goles").alias("PromedioGoles"),
    sum("Pases").alias("TotalPases"),
    max("Velocidad").alias("MaxVelocidad")
).show()


## 🔗 Combinación de DataFrames

In [0]:

# Crear otro DataFrame con nuevos jugadores
datos_extra = [
    ("Bellingham", "Real Madrid", 31.5, 4, 80, 70, 90),
    ("Ferran", "FC Barcelona", 30.8, 2, 65, 50, 75),
    ("Isco", "Real Betis", 28.7, 2, 90, 75, 83)
]
df2 = spark.createDataFrame(datos_extra, columns)

In [0]:
# Unión (añadimos al primer conjunto de jugadores el segundo)
df.union(df2).show()

In [0]:
# Eliminar duplicados (Isco aparece en ambos DataFrames)
df.union(df2).distinct().show()

In [0]:
# Intersección (devuelve los jugadores que aparecen en ambos)
df.intersect(df2).show()

In [0]:
# Diferencia (devuelve los jugadores del primero quitando los jugadores del segundo)
df.subtract(df2).show()

In [0]:
# DataFrame de equipos
teams_data = [
    ("FC Barcelona", "Xavi", 1),
    ("Real Madrid", "Ancelotti", 2),
    ("Real Sociedad", "Alguacil", 3),
    ("Real Betis", "Pellegrini", 4)
]
columns_teams = ["Equipo", "Entrenador", "GrupoChampions"]
df_teams = spark.createDataFrame(teams_data, columns_teams)

# Join por columna 'Equipo'
df_complete = df.join(df_teams, on="Equipo", how="inner")

# Mostrar resultado
df_complete.show()

## 🧹 Orden y organización

In [0]:
# Ordenar por goles descendente
df.orderBy(df["Goles"].desc()).show()

## 📁 Lectura y escritura

In [0]:
# Guardar los resultados (en disco o en un sistema de ficheros distribuido)
df.write.csv("/Volumes/workspace/liga/datos/output_players", header=True, mode="overwrite")

# Podemos agrupar los datos en una única partición usando coalesce(1)
# df.coalesce(1).write.csv("/Volumes/workspace/liga/datos/output_players", header=True, mode="overwrite")

# Leer CSV (requiere archivo existente)
df_players = spark.read.csv("/Volumes/workspace/liga/datos/output_players", header=True, inferSchema=True)
df_players.show()

## 🧠 Consultas estructuradas

Es posible realizar consultas utilizando lenguaje SQL sobre el conjunto de datos. Se crea una vista temporal llamada 'jugadores' que actuará como una tabla.

In [0]:
df.createOrReplaceTempView("jugadores")
sql_df = spark.sql("SELECT Equipo, AVG(Goles) as PromedioGoles FROM jugadores GROUP BY Equipo")
sql_df.show()

## 🔎 Exploración y depuración

Tenemos a nuestra disposición diferentes funciones y propiedades para explorar en detalle los datos

In [0]:
# Muestra los datos como una tabla
df.show()

In [0]:
# Muestra el esquema de los datos (los tipos de las columnas)
df.printSchema()

In [0]:
# Muestra estadísticos de los datos
df.describe().show()

In [0]:
# Obtiene las columnas como una lista
df.columns

In [0]:
# Obtiene un array de tuplas (Columna, Tipo)
df.dtypes